In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
train = pd.read_csv("train_data.csv")

Train-Validation Split

Remove employee_id as it dont give any information

In [3]:
# Separate features and target
X = train.drop(columns=['employee_id','is_promoted'])
y = train['is_promoted'].copy()

# Split the data (80% train, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)

Training shape: (43846, 12)
Validation shape: (10962, 12)


Missing Value Handling

In [4]:
# Fill missing values in 'education' with "Unknown"
X_train['education'] = X_train['education'].fillna('Unknown')
X_val['education'] = X_val['education'].fillna('Unknown')

# Fill missing values in 'previous_year_rating' with 0
if 'previous_year_rating' in X_train.columns:
    X_train.loc[:, 'previous_year_rating'] = X_train['previous_year_rating'].fillna(0)
    X_val.loc[:, 'previous_year_rating'] = X_val['previous_year_rating'].fillna(0)

In [5]:
X_train.isnull().sum()

department              0
region                  0
education               0
gender                  0
recruitment_channel     0
no_of_trainings         0
age                     0
previous_year_rating    0
length_of_service       0
KPIs_met >80%           0
awards_won?             0
avg_training_score      0
dtype: int64

Feature Lists

In [5]:
num_feats = ['no_of_trainings', 'age', 'previous_year_rating', 'length_of_service', 'avg_training_score']
cat_feats = ['region', 'education', 'gender','department','recruitment_channel']

print("Numeric features:", num_feats)
print("Categorical features:", cat_feats)

Numeric features: ['no_of_trainings', 'age', 'previous_year_rating', 'length_of_service', 'avg_training_score']
Categorical features: ['region', 'education', 'gender', 'department', 'recruitment_channel']


Encoding categorical columns as they can not be directly processed by most of the models

In [6]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

cat_cols = [
    'department',
    'region',
    'education',
    'gender',
    'recruitment_channel'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=False
        ), cat_cols)
    ],
    remainder='passthrough'
)

# Fit only on training data
X_train_encoded = preprocessor.fit_transform(X_train)

# Apply the same encoding to validation data
X_val_encoded = preprocessor.transform(X_val)

print("X_train shape:", X_train_encoded.shape)
print("X_val shape:", X_val_encoded.shape)


X_train shape: (43846, 59)
X_val shape: (10962, 59)


Baseline Model

In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Baseline Decision Tree
dt = DecisionTreeClassifier(
    random_state=42
)

# Train on encoded data
dt.fit(X_train_encoded, y_train)

# Predict on encoded validation data
y_pred = dt.predict(X_val_encoded)

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))

print("\nClassification Report:")
print(classification_report(y_val, y_pred))

Confusion Matrix:
[[9461  567]
 [ 504  430]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.94      0.95     10028
           1       0.43      0.46      0.45       934

    accuracy                           0.90     10962
   macro avg       0.69      0.70      0.70     10962
weighted avg       0.91      0.90      0.90     10962



In [12]:
feature_names = preprocessor.get_feature_names_out()

In [13]:
import pandas as pd

feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': dt.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='importance',
    ascending=False
)

print(feature_importance.head(20))

                              feature  importance
58      remainder__avg_training_score    0.252598
53                     remainder__age    0.117703
55       remainder__length_of_service    0.078212
7   cat__department_Sales & Marketing    0.060782
54    remainder__previous_year_rating    0.049953
56           remainder__KPIs_met >80%    0.042093
4          cat__department_Operations    0.037396
52         remainder__no_of_trainings    0.023902
5         cat__department_Procurement    0.023650
57             remainder__awards_won?    0.020407
51  cat__recruitment_channel_sourcing    0.019498
49     cat__recruitment_channel_other    0.019131
0           cat__department_Analytics    0.016918
47                      cat__gender_f    0.015197
48                      cat__gender_m    0.013572
1             cat__department_Finance    0.012826
8          cat__department_Technology    0.012226
20               cat__region_region_2    0.011726
23              cat__region_region_22    0.011485


Lets try improving decision tree by using hyperparameter tuning

In [14]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV

dt = DecisionTreeClassifier(random_state=42)

param_dist = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [3, 5, 7, 9, 11, 13, 15, 20, None],
    'min_samples_split': [2, 5, 10, 20, 30, 50],
    'min_samples_leaf': [1, 2, 5, 10, 20, 30],
    'max_features': [None, 'sqrt', 'log2'],
    'class_weight': [None, 'balanced']
}

random_search = RandomizedSearchCV(
    estimator=dt,
    param_distributions=param_dist,
    n_iter=50,
    scoring='f1',
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train_encoded, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


,estimator,DecisionTreeC...ndom_state=42)
,param_distributions,"{'class_weight': [None, 'balanced'], 'criterion': ['gini', 'entropy', ...], 'max_depth': [3, 5, ...], 'max_features': [None, 'sqrt', ...], ...}"
,n_iter,50
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [16]:
print("Best Parameters:")
print(random_search.best_params_)

print("\nBest CV F1 Score:")
print(random_search.best_score_)

Best Parameters:
{'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': 20, 'criterion': 'gini', 'class_weight': None}

Best CV F1 Score:
0.4808647978884002


In [15]:
best_dt = random_search.best_estimator_

In [17]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

y_pred_tuned = best_dt.predict(X_val_encoded)
y_prob_tuned = best_dt.predict_proba(X_val_encoded)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_tuned))

print("\nClassification Report:")
print(classification_report(y_val, y_pred_tuned))

print("ROC-AUC:", roc_auc_score(y_val, y_prob_tuned))

Confusion Matrix:
[[9841  187]
 [ 563  371]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.98      0.96     10028
           1       0.66      0.40      0.50       934

    accuracy                           0.93     10962
   macro avg       0.81      0.69      0.73     10962
weighted avg       0.92      0.93      0.92     10962

ROC-AUC: 0.8159217894392489


Handling class imbalance and threshold

In [18]:
import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

# ============================================================
# 1. BASE DECISION TREE
# ============================================================

dt = DecisionTreeClassifier(
    random_state=42
)


# ============================================================
# 2. HYPERPARAMETER SEARCH
# ============================================================

param_dist = {
    'criterion': ['gini', 'entropy', 'log_loss'],

    'max_depth': [3, 5, 7, 9, 11, 13, 15, 20, None],

    'min_samples_split': [2, 5, 10, 20, 30, 50],

    'min_samples_leaf': [1, 2, 5, 10, 20, 30],

    'max_features': [None, 'sqrt', 'log2'],

    # Custom handling of class imbalance
    'class_weight': [
        None,
        {0: 1, 1: 2},
        {0: 1, 1: 3},
        {0: 1, 1: 4},
        {0: 1, 1: 5}
    ]
}


# ============================================================
# 3. RANDOMIZED SEARCH
# ============================================================

random_search = RandomizedSearchCV(
    estimator=dt,
    param_distributions=param_dist,
    n_iter=50,
    scoring='f1',
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(
    X_train_encoded,
    y_train
)


# ============================================================
# 4. BEST MODEL
# ============================================================

best_dt = random_search.best_estimator_

print("\nBest Parameters:")
print(random_search.best_params_)

print("\nBest Cross-Validation F1:")
print(random_search.best_score_)


# ============================================================
# 5. GET PROBABILITIES
# ============================================================

y_prob = best_dt.predict_proba(X_val_encoded)[:, 1]

print("\nROC-AUC:")
print(roc_auc_score(y_val, y_prob))


# ============================================================
# 6. FIND BEST THRESHOLD USING F1
# ============================================================

thresholds = np.arange(0.10, 0.91, 0.01)

results = []

for threshold in thresholds:

    y_pred_threshold = (y_prob >= threshold).astype(int)

    precision = precision_score(
        y_val,
        y_pred_threshold,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_pred_threshold,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_pred_threshold,
        zero_division=0
    )

    results.append({
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })


threshold_results = pd.DataFrame(results)

# Best threshold based on F1
best_result = threshold_results.loc[
    threshold_results['f1'].idxmax()
]

best_threshold = best_result['threshold']

print("\nBest Threshold:")
print(round(best_threshold, 2))

print("\nPerformance at Best Threshold:")
print(best_result)


# ============================================================
# 7. FINAL PREDICTIONS USING BEST THRESHOLD
# ============================================================

y_pred_final = (
    y_prob >= best_threshold
).astype(int)


# ============================================================
# 8. FINAL EVALUATION
# ============================================================

print("\n" + "=" * 50)
print("FINAL DECISION TREE RESULTS")
print("=" * 50)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_final))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_pred_final
))

print("ROC-AUC:",
      round(roc_auc_score(y_val, y_prob), 4))

print("Precision:",
      round(precision_score(y_val, y_pred_final), 4))

print("Recall:",
      round(recall_score(y_val, y_pred_final), 4))

print("F1 Score:",
      round(f1_score(y_val, y_pred_final), 4))

Fitting 5 folds for each of 50 candidates, totalling 250 fits

Best Parameters:
{'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': 13, 'criterion': 'log_loss', 'class_weight': {0: 1, 1: 3}}

Best Cross-Validation F1:
0.48823638938770986

ROC-AUC:
0.8619225376654149

Best Threshold:
0.61

Performance at Best Threshold:
threshold    0.610000
precision    0.683636
recall       0.402570
f1           0.506739
Name: 51, dtype: float64

FINAL DECISION TREE RESULTS

Confusion Matrix:
[[9854  174]
 [ 558  376]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.98      0.96     10028
           1       0.68      0.40      0.51       934

    accuracy                           0.93     10962
   macro avg       0.82      0.69      0.74     10962
weighted avg       0.92      0.93      0.93     10962

ROC-AUC: 0.8619
Precision: 0.6836
Recall: 0.4026
F1 Score: 0.5067


I think Decision tree has reached its limit.

In [19]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_encoded, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [20]:
y_pred_rf = rf.predict(X_val_encoded)
y_prob_rf = rf.predict_proba(X_val_encoded)[:, 1]

In [21]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_val, y_pred_rf))

print("ROC-AUC:", roc_auc_score(y_val, y_prob_rf))

Confusion Matrix:
[[9967   61]
 [ 682  252]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.99      0.96     10028
           1       0.81      0.27      0.40       934

    accuracy                           0.93     10962
   macro avg       0.87      0.63      0.68     10962
weighted avg       0.92      0.93      0.92     10962

ROC-AUC: 0.8818622097954422


In [22]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

thresholds = np.arange(0.10, 0.91, 0.01)

results = []

for threshold in thresholds:

    y_pred = (y_prob_rf >= threshold).astype(int)

    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)

    results.append({
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

threshold_results = pd.DataFrame(results)

# Best threshold according to F1
best_row = threshold_results.loc[
    threshold_results['f1'].idxmax()
]

print("Best threshold:")
print(round(best_row['threshold'], 2))

print("\nPerformance at best threshold:")
print(best_row)

Best threshold:
0.31

Performance at best threshold:
threshold    0.310000
precision    0.526797
recall       0.431478
f1           0.474397
Name: 21, dtype: float64


Hyperparameter tuning,Handling class imbalance and threshold optimization

In [24]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

param_dist = {
    'n_estimators': [100, 150, 200],
    'max_depth': [8, 12, 16, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': ['sqrt', 'log2'],
    'class_weight': [
        None,
        {0: 1, 1: 2},
        {0: 1, 1: 3}
    ]
}

random_search_rf = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=15,          # only 15 combinations
    scoring='f1',
    cv=3,               # 3-fold instead of 5
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search_rf.fit(
    X_train_encoded,
    y_train
)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'class_weight': [None, {0: 1, 1: 2}, ...], 'max_depth': [8, 12, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, ...], ...}"
,n_iter,15
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,3
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [25]:
best_rf = random_search_rf.best_estimator_

print("Best Parameters:")
print(random_search_rf.best_params_)

print("\nBest CV F1:")
print(random_search_rf.best_score_)

Best Parameters:
{'n_estimators': 150, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None, 'class_weight': {0: 1, 1: 3}}

Best CV F1:
0.3926900004060469


In [26]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

y_prob_rf_tuned = best_rf.predict_proba(X_val_encoded)[:, 1]

# Default threshold = 0.5
y_pred_rf_tuned = (y_prob_rf_tuned >= 0.5).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_rf_tuned))

print("\nClassification Report:")
print(classification_report(y_val, y_pred_rf_tuned))

print("ROC-AUC:",
      round(roc_auc_score(y_val, y_prob_rf_tuned), 4))

Confusion Matrix:
[[9943   85]
 [ 671  263]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.99      0.96     10028
           1       0.76      0.28      0.41       934

    accuracy                           0.93     10962
   macro avg       0.85      0.64      0.69     10962
weighted avg       0.92      0.93      0.92     10962

ROC-AUC: 0.892


In [27]:
y_prob_rf_tuned

array([0.14238963, 0.05157992, 0.009     , ..., 0.32485127, 0.2777491 ,
       0.01417605], shape=(10962,))

In [28]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

thresholds = np.arange(0.10, 0.71, 0.01)

results = []

for threshold in thresholds:

    y_pred = (y_prob_rf_tuned >= threshold).astype(int)

    precision = precision_score(
        y_val, y_pred, zero_division=0
    )

    recall = recall_score(
        y_val, y_pred, zero_division=0
    )

    f1 = f1_score(
        y_val, y_pred, zero_division=0
    )

    results.append({
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

threshold_results_rf = pd.DataFrame(results)

# Best threshold based on F1
best_row_rf = threshold_results_rf.loc[
    threshold_results_rf['f1'].idxmax()
]

print("Best Threshold:", round(best_row_rf['threshold'], 2))
print("\nBest Performance:")
print(best_row_rf)

Best Threshold: 0.36

Best Performance:
threshold    0.360000
precision    0.474359
recall       0.475375
f1           0.474866
Name: 26, dtype: float64


Try Boosting

In [7]:
from xgboost import XGBClassifier

In [8]:
#calculating class imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print("Scale Pos Weight:", scale_pos_weight)

Scale Pos Weight: 10.742367434386717


In [9]:
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

xgb.fit(
    X_train_encoded,
    y_train
)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [10]:
y_pred_xgb = xgb.predict(X_val_encoded)

y_prob_xgb = xgb.predict_proba(
    X_val_encoded
)[:, 1]

In [11]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_xgb))

print("\nClassification Report:")
print(classification_report(y_val, y_pred_xgb))

print(
    "ROC-AUC:",
    round(roc_auc_score(y_val, y_prob_xgb), 4)
)

Confusion Matrix:
[[7363 2665]
 [  90  844]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.73      0.84     10028
           1       0.24      0.90      0.38       934

    accuracy                           0.75     10962
   macro avg       0.61      0.82      0.61     10962
weighted avg       0.92      0.75      0.80     10962

ROC-AUC: 0.9108


lets tune the model

In [34]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

thresholds = np.arange(0.10, 0.91, 0.01)

results_xgb = []

for threshold in thresholds:

    y_pred = (y_prob_xgb >= threshold).astype(int)

    precision = precision_score(
        y_val,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_pred,
        zero_division=0
    )

    results_xgb.append({
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

threshold_results_xgb = pd.DataFrame(results_xgb)

best_xgb = threshold_results_xgb.loc[
    threshold_results_xgb['f1'].idxmax()
]

print("Best Threshold:", round(best_xgb['threshold'], 2))

print("\nBest Performance:")
print(best_xgb)

Best Threshold: 0.76

Best Performance:
threshold    0.760000
precision    0.660194
recall       0.436831
f1           0.525773
Name: 66, dtype: float64


In [35]:
import numpy as np

from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

# Class imbalance
scale_pos_weight = (
    (y_train == 0).sum() /
    (y_train == 1).sum()
)

xgb = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

param_dist = {
    'n_estimators': [100, 150, 200, 250, 300],
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.03, 0.05, 0.08, 0.1],
    'min_child_weight': [1, 3, 5, 7],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

random_search_xgb = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=20,
    scoring='f1',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search_xgb.fit(
    X_train_encoded,
    y_train
)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


,estimator,"XGBClassifier...ree=None, ...)"
,param_distributions,"{'colsample_bytree': [0.7, 0.8, ...], 'learning_rate': [0.03, 0.05, ...], 'max_depth': [3, 4, ...], 'min_child_weight': [1, 3, ...], ...}"
,n_iter,20
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,3
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [36]:
print("Best Parameters:")
print(random_search_xgb.best_params_)

print("\nBest CV F1:")
print(random_search_xgb.best_score_)

Best Parameters:
{'subsample': 0.7, 'n_estimators': 250, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.8}

Best CV F1:
0.4295255318377731


In [37]:
best_xgb = random_search_xgb.best_estimator_

In [39]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

y_prob_xgb_tuned = best_xgb.predict_proba(
    X_val_encoded
)[:, 1]

y_pred_xgb_tuned = (
    y_prob_xgb_tuned >= 0.5
).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_xgb_tuned))

print("\nClassification Report:")
print(classification_report(y_val, y_pred_xgb_tuned))

print(
    "ROC-AUC:",
    round(
        roc_auc_score(y_val, y_prob_xgb_tuned),
        4
    )
)

Confusion Matrix:
[[8141 1887]
 [ 194  740]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.81      0.89     10028
           1       0.28      0.79      0.42       934

    accuracy                           0.81     10962
   macro avg       0.63      0.80      0.65     10962
weighted avg       0.92      0.81      0.85     10962

ROC-AUC: 0.9086


In [40]:
y_prob_xgb_tuned

array([0.255739  , 0.00227977, 0.00098619, ..., 0.39046064, 0.83979577,
       0.01870911], shape=(10962,), dtype=float32)

In [41]:

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

thresholds = np.arange(0.30, 0.91, 0.01)

results_xgb_tuned = []

for threshold in thresholds:

    y_pred = (
        y_prob_xgb_tuned >= threshold
    ).astype(int)

    precision = precision_score(
        y_val,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_pred,
        zero_division=0
    )

    results_xgb_tuned.append({
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

threshold_results_xgb_tuned = pd.DataFrame(
    results_xgb_tuned
)

best_xgb_threshold = threshold_results_xgb_tuned.loc[
    threshold_results_xgb_tuned['f1'].idxmax()
]

print("Best Threshold:")
print(round(best_xgb_threshold['threshold'], 2))

print("\nBest Performance:")
print(best_xgb_threshold)

Best Threshold:
0.81

Best Performance:
threshold    0.810000
precision    0.676370
recall       0.422912
f1           0.520422
Name: 51, dtype: float64


In [42]:
threshold_results_xgb_tuned.sort_values(
    by='f1',
    ascending=False
).head(10)

,threshold,precision,recall,f1
51,0.81,0.676370,0.422912,0.520422
52,0.82,0.705128,0.412206,0.520270
55,0.85,0.795556,0.383298,0.517341
53,0.83,0.731373,0.399358,0.516620
54,0.84,0.757764,0.391863,0.516584
50,0.80,0.631988,0.435760,0.515843
48,0.78,0.582210,0.462527,0.515513
49,0.79,0.608187,0.445396,0.514215
47,0.77,0.553903,0.478587,0.513498
56,0.86,0.814988,0.372591,0.511389


Catboost

In [14]:
from catboost import CatBoostClassifier

In [15]:
cat_cols = [
    'department',
    'region',
    'education',
    'gender',
    'recruitment_channel'
]

In [16]:
X_train_cb = X_train.copy()
X_val_cb = X_val.copy()

for col in cat_cols:
    X_train_cb[col] = X_train_cb[col].astype(str)
    X_val_cb[col] = X_val_cb[col].astype(str)

Base Model

In [46]:
cat_model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=100
)

cat_model.fit(
    X_train_cb,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_val_cb, y_val),
    early_stopping_rounds=50
)

0:	test: 0.7898405	best: 0.7898405 (0)	total: 321ms	remaining: 1m 35s
100:	test: 0.9063212	best: 0.9063212 (100)	total: 13.9s	remaining: 27.3s
200:	test: 0.9120875	best: 0.9120875 (200)	total: 27.3s	remaining: 13.4s
299:	test: 0.9132674	best: 0.9133332 (297)	total: 39.7s	remaining: 0us

bestTest = 0.9133332451
bestIteration = 297

Shrink model to first 298 iterations.


In [47]:
y_pred_cb = cat_model.predict(X_val_cb).ravel()

y_prob_cb = cat_model.predict_proba(
    X_val_cb
)[:, 1]

In [48]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_cb))

print("\nClassification Report:")
print(classification_report(y_val, y_pred_cb))

print(
    "ROC-AUC:",
    round(roc_auc_score(y_val, y_prob_cb), 4)
)

Confusion Matrix:
[[10018    10]
 [  618   316]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      1.00      0.97     10028
           1       0.97      0.34      0.50       934

    accuracy                           0.94     10962
   macro avg       0.96      0.67      0.74     10962
weighted avg       0.94      0.94      0.93     10962

ROC-AUC: 0.9133


In [49]:
y_prob_cb

array([0.09318259, 0.00182964, 0.00062416, ..., 0.08718348, 0.26090257,
       0.00768388], shape=(10962,))

Trying another thresholds

In [50]:

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

thresholds = np.arange(0.10, 0.91, 0.01)

results_cb = []

for threshold in thresholds:

    y_pred = (
        y_prob_cb >= threshold
    ).astype(int)

    precision = precision_score(
        y_val,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_pred,
        zero_division=0
    )

    results_cb.append({
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

threshold_results_cb = pd.DataFrame(results_cb)

best_cb = threshold_results_cb.loc[
    threshold_results_cb['f1'].idxmax()
]

print("Best Threshold:", round(best_cb['threshold'], 2))

print("\nBest Performance:")
print(best_cb)

Best Threshold: 0.23

Best Performance:
threshold    0.230000
precision    0.649311
recall       0.453961
f1           0.534342
Name: 13, dtype: float64


In [52]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Best threshold from our search
best_threshold_cb = 0.23

# Convert probabilities into predictions
y_pred_cb_final = (
    y_prob_cb >= best_threshold_cb
).astype(int)

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_cb_final))

# Classification Report
print("\nClassification Report:")
print(classification_report(
    y_val,
    y_pred_cb_final
))

# Individual metrics
print("Precision:",
      round(precision_score(y_val, y_pred_cb_final), 4))

print("Recall:",
      round(recall_score(y_val, y_pred_cb_final), 4))

print("F1 Score:",
      round(f1_score(y_val, y_pred_cb_final), 4))

print("ROC-AUC:",
      round(roc_auc_score(y_val, y_prob_cb), 4))

Confusion Matrix:
[[9799  229]
 [ 510  424]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.98      0.96     10028
           1       0.65      0.45      0.53       934

    accuracy                           0.93     10962
   macro avg       0.80      0.72      0.75     10962
weighted avg       0.92      0.93      0.93     10962

Precision: 0.6493
Recall: 0.454
F1 Score: 0.5343
ROC-AUC: 0.9133


Trying different values of parameters

In [17]:
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, roc_auc_score

models = {
    "CatBoost_1": {
        "depth": 4,
        "learning_rate": 0.05,
        "l2_leaf_reg": 5
    },

    "CatBoost_2": {
        "depth": 5,
        "learning_rate": 0.05,
        "l2_leaf_reg": 5
    },

    "CatBoost_3": {
        "depth": 6,
        "learning_rate": 0.05,
        "l2_leaf_reg": 10
    },

    "CatBoost_4": {
        "depth": 5,
        "learning_rate": 0.03,
        "l2_leaf_reg": 10
    },

    "CatBoost_5": {
        "depth": 6,
        "learning_rate": 0.03,
        "l2_leaf_reg": 20
    }
}

results = []

for name, params in models.items():

    print(f"\nTraining {name}...")

    model = CatBoostClassifier(
        iterations=500,
        depth=params["depth"],
        learning_rate=params["learning_rate"],
        l2_leaf_reg=params["l2_leaf_reg"],
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False,
        thread_count=-1,
        allow_writing_files=False
    )

    model.fit(
        X_train_cb,
        y_train,
        cat_features=cat_cols,
        eval_set=(X_val_cb, y_val),
        early_stopping_rounds=50,
        verbose=False
    )

    prob = model.predict_proba(X_val_cb)[:, 1]

    auc = roc_auc_score(y_val, prob)

    # Use the same threshold we found for our original CatBoost
    pred = (prob >= 0.23).astype(int)

    f1 = f1_score(y_val, pred)

    results.append({
        "model": name,
        "AUC": auc,
        "F1": f1,
        "best_iteration": model.get_best_iteration()
    })

results_df = pd.DataFrame(results)

print("\nResults:")
print(results_df.sort_values("F1", ascending=False))


Training CatBoost_1...

Training CatBoost_2...

Training CatBoost_3...

Training CatBoost_4...

Training CatBoost_5...

Results:
        model       AUC        F1  best_iteration
2  CatBoost_3  0.914188  0.534230             475
4  CatBoost_5  0.912876  0.533509             389
1  CatBoost_2  0.914578  0.532595             430
0  CatBoost_1  0.913332  0.527900             495
3  CatBoost_4  0.911923  0.525490             455


In [18]:
# Train the selected CatBoost model

best_cb = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=10,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=False,
    thread_count=-1,
    allow_writing_files=False
)

best_cb.fit(
    X_train_cb,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_val_cb, y_val),
    early_stopping_rounds=50,
    verbose=False
)

# Get probabilities
y_prob_best_cb = best_cb.predict_proba(
    X_val_cb
)[:, 1]

print("Best iteration:", best_cb.get_best_iteration())

Best iteration: 475


In [19]:
y_prob_best_cb 

array([0.0865984 , 0.00193402, 0.00069827, ..., 0.08210015, 0.30966648,
       0.00629402], shape=(10962,))

In [20]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

thresholds = np.arange(0.10, 0.91, 0.01)

results_cb = []

for threshold in thresholds:

    y_pred = (
        y_prob_best_cb >= threshold
    ).astype(int)

    precision = precision_score(
        y_val,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_pred,
        zero_division=0
    )

    results_cb.append({
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

threshold_results_cb = pd.DataFrame(results_cb)

# Find threshold with highest F1
best_row_cb = threshold_results_cb.loc[
    threshold_results_cb['f1'].idxmax()
]

print("Best Threshold:")
print(round(best_row_cb['threshold'], 2))

print("\nBest Performance:")
print(best_row_cb)

Best Threshold:
0.22

Best Performance:
threshold    0.220000
precision    0.598465
recall       0.501071
f1           0.545455
Name: 12, dtype: float64


We got our best model

In [58]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred_final = (
    y_prob_best_cb >= 0.22
).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_final))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_pred_final
))

Confusion Matrix:
[[9714  314]
 [ 466  468]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.97      0.96     10028
           1       0.60      0.50      0.55       934

    accuracy                           0.93     10962
   macro avg       0.78      0.73      0.75     10962
weighted avg       0.92      0.93      0.93     10962



In [14]:
# Get feature importance
importance = best_cb.get_feature_importance()

# Create dataframe
feature_importance = pd.DataFrame({
    'feature': X_train_cb.columns,
    'importance': importance
})

# Sort from highest to lowest
feature_importance = feature_importance.sort_values(
    by='importance',
    ascending=False
).reset_index(drop=True)

print(feature_importance)

                 feature  importance
0          KPIs_met >80%   34.876111
1     avg_training_score   34.542535
2             department   12.435907
3   previous_year_rating    7.676502
4      length_of_service    2.176120
5                 region    2.051426
6                    age    2.011135
7            awards_won?    1.839760
8              education    0.930501
9        no_of_trainings    0.647063
10   recruitment_channel    0.598216
11                gender    0.214723


model outcomes exploration

In [56]:
# Create validation results
val_results = X_val_cb.copy()

val_results["actual"] = y_val.values
val_results["probability"] = y_prob_best_cb
val_results["prediction"] = (
    y_prob_best_cb >= 0.22
).astype(int)

# False negatives:
# Actually promoted, but model predicted not promoted
false_negatives = val_results[
    (val_results["actual"] == 1) &
    (val_results["prediction"] == 0)
]

print("Number of false negatives:", len(false_negatives))

print("\nFalse negative profile:")
print(
    false_negatives[
        [
            "department",
            "education",
            "gender",
            "recruitment_channel",
            "no_of_trainings",
            "age",
            "previous_year_rating",
            "length_of_service",
            "KPIs_met >80%",
            "awards_won?",
            "avg_training_score"
        ]
    ].describe(include="all")
)

Number of false negatives: 466

False negative profile:
               department   education gender recruitment_channel  \
count                 466         466    466                 466   
unique                  9           4      2                   3   
top     Sales & Marketing  Bachelor's      m               other   
freq                  110         293    313                 259   
mean                  NaN         NaN    NaN                 NaN   
std                   NaN         NaN    NaN                 NaN   
min                   NaN         NaN    NaN                 NaN   
25%                   NaN         NaN    NaN                 NaN   
50%                   NaN         NaN    NaN                 NaN   
75%                   NaN         NaN    NaN                 NaN   
max                   NaN         NaN    NaN                 NaN   

        no_of_trainings         age  previous_year_rating  length_of_service  \
count        466.000000  466.000000            

In [57]:
# Compare all actual promoted employees
# with the false negatives

promoted = val_results[
    val_results["actual"] == 1
]

numeric_cols = [
    "no_of_trainings",
    "age",
    "previous_year_rating",
    "length_of_service",
    "KPIs_met >80%",
    "awards_won?",
    "avg_training_score"
]

comparison = pd.DataFrame({
    "All Promoted": promoted[numeric_cols].mean(),
    "False Negatives": false_negatives[numeric_cols].mean()
})

print(comparison)

                      All Promoted  False Negatives
no_of_trainings           1.211991         1.214592
age                      34.618844        34.437768
previous_year_rating      3.699143         3.667382
length_of_service         5.793362         5.386266
KPIs_met >80%             0.689507         0.896996
awards_won?               0.137045         0.025751
avg_training_score       71.639186        65.680258


In [58]:
categorical_cols = [
    "department",
    "education",
    "gender",
    "recruitment_channel"
]

for col in categorical_cols:
    print("\n" + "="*50)
    print(col)

    comparison_cat = pd.DataFrame({
        "All Promoted %": promoted[col].value_counts(normalize=True) * 100,
        "False Negative %": false_negatives[col].value_counts(normalize=True) * 100
    }).fillna(0)

    print(comparison_cat.round(2))


department
                   All Promoted %  False Negative %
department                                         
Analytics                   11.88             17.38
Finance                      5.35              4.51
HR                           3.10              3.65
Legal                        0.86              0.86
Operations                  20.88             20.39
Procurement                 14.78             13.95
R&D                          0.86              1.29
Sales & Marketing           24.41             23.61
Technology                  17.88             14.38

education
                  All Promoted %  False Negative %
education                                         
Bachelor's                 61.56             62.88
Master's & above           34.15             32.83
Unknown                     2.89              2.58
Below Secondary             1.39              1.72

gender
        All Promoted %  False Negative %
gender                                  
m        